# Batch F5-TTS Vietnamese 1000h từ SRT

Notebook này tạo audio tiếng Việt từ các file `.srt` bằng `hynt/F5-TTS-Vietnamese-ViVoice` trên Colab GPU.

Upload cùng lúc:
- Một hoặc nhiều file `.srt` tiếng Việt.
- Một file audio tham chiếu giọng đọc, ví dụ `audio-truyen.mp3`.
- Script `scripts/colab_batch_f5tts_vietnamese_from_srt.py` từ repo này.

Model dùng license non-commercial/research. Chỉ clone giọng khi bạn có quyền hoặc có sự đồng ý rõ ràng.

In [ ]:
REF_TEXT = ""  # Nên điền transcript đúng của audio tham chiếu nếu có. Để trống thì F5-TTS sẽ tự transcribe.
REFERENCE_START = 0.0
REFERENCE_DURATION = 12.0
TIMING_MODE = "no_cut_sequential"  # "no_cut_sequential" hoặc "fit_segments"
MAX_TEMPO = 1.35
SPEED = 1.0
NFE_STEP = 32
CFG_STRENGTH = 2.0
SWAY_SAMPLING_COEF = -1.0
MODEL_REPO = "hynt/F5-TTS-Vietnamese-ViVoice"
MERGE_SCOPE = "all"  # "all" = mọi SRT thành 1 audio; "per_srt" = mỗi SRT thành 1 audio

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install -U f5-tts huggingface_hub soundfile pydub cached_path hydra-core omegaconf faster-whisper

In [ ]:
from google.colab import files
from pathlib import Path
import os, shutil

work = Path("/content/f5tts_job")
srt_dir = work / "srt"
output_dir = Path("/content/f5tts_vietnamese_audio_results")
srt_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
script_path = None
ref_audio = None

for name in uploaded:
    src = Path(name)
    suffix = src.suffix.lower()
    if suffix == ".srt":
        shutil.move(str(src), srt_dir / src.name)
    elif src.name == "colab_batch_f5tts_vietnamese_from_srt.py" or suffix == ".py":
        script_path = work / "colab_batch_f5tts_vietnamese_from_srt.py"
        shutil.move(str(src), script_path)
    elif suffix in {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".aac"}:
        ref_audio = work / src.name
        shutil.move(str(src), ref_audio)

if script_path is None:
    raise RuntimeError("Chưa upload scripts/colab_batch_f5tts_vietnamese_from_srt.py")
if ref_audio is None:
    raise RuntimeError("Chưa upload audio tham chiếu giọng đọc")
if not list(srt_dir.glob("*.srt")):
    raise RuntimeError("Chưa upload file .srt")

print("SRT dir:", srt_dir)
print("Reference audio:", ref_audio)
print("Script:", script_path)

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, str(script_path),
    "--srt-dir", str(srt_dir),
    "--output-dir", str(output_dir),
    "--ref-audio", str(ref_audio),
    "--ref-text", REF_TEXT,
    "--model-repo", MODEL_REPO,
    "--timing-mode", TIMING_MODE,
    "--max-tempo", str(MAX_TEMPO),
    "--reference-start", str(REFERENCE_START),
    "--reference-duration", str(REFERENCE_DURATION),
    "--speed", str(SPEED),
    "--nfe-step", str(NFE_STEP),
    "--cfg-strength", str(CFG_STRENGTH),
    "--sway-sampling-coef", str(SWAY_SAMPLING_COEF),
    "--merge-all-text",
    "--merge-scope", MERGE_SCOPE,
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
from google.colab import files
zip_path = "/content/f5tts_vietnamese_audio_results.zip"
files.download(zip_path)